# Week 09 | Patterns, prediction and data leakage

**Core practical: 45 minutes.** D2: reveal donor leakage with a simple model and a defensible evaluation unit.

No paid AI tool, local installation or external dataset download is required. In Colab, upload this notebook through File > Upload notebook, then run cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

**Project milestone:** D2 submission: notebook, <=1200-word evaluation memo, split diagram/table, metrics and limitations, GENOMES audit and AI/no-AI disclosure. Due end Week 10; 12% of course. No advanced neural network required.

## Before running (5 min)
A feature is a measured input; a label is the outcome being predicted. Rows from the same donor can share a signature.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Read the D2 scenario and predict which split is easier.
2. Compare within-donor row splitting with a new-donor holdout.
3. Report row and donor metrics, donor counts and the fitted scaling boundary.
4. Run donor-level label permutations as a reference distribution, not a clinical validation.

In [ ]:
# SYNTHETIC: labels are assigned randomly to donor fingerprints, with no population-level disease signal.
N_DONORS, ROWS_PER_DONOR, FEATURES = 12, 6, 8
labels = [0]*6 + [1]*6
rng.shuffle(labels)
rows = []
for donor in range(N_DONORS):
    signature = [rng.gauss(0,3) for _ in range(FEATURES)]
    for repeat in range(ROWS_PER_DONOR):
        rows.append({"donor":donor,"repeat":repeat,"y":labels[donor],
                     "x":[v+rng.gauss(0,0.12) for v in signature]})
row_train = [r for r in rows if r["repeat"] < 4]
row_test = [r for r in rows if r["repeat"] >= 4]
# Stratify the pedagogic holdout by donor label; do not tune after inspecting test predictions.
holdout = set([i for i,y in enumerate(labels) if y == 0][-2:] +
              [i for i,y in enumerate(labels) if y == 1][-2:])
group_train = [r for r in rows if r["donor"] not in holdout]
group_test = [r for r in rows if r["donor"] in holdout]

def predict_1nn(train, test, label_map=None):
    # Standardization learned only on training rows, then applied unchanged to test rows.
    mu = [mean([r["x"][j] for r in train]) for j in range(FEATURES)]
    sd = [math.sqrt(mean([(r["x"][j]-mu[j])**2 for r in train])) or 1 for j in range(FEATURES)]
    def distance(a,b): return sum(((a[j]-b[j])/sd[j])**2 for j in range(FEATURES))
    predictions=[]
    for row in test:
        near = min(train,key=lambda r:distance(r["x"],row["x"]))
        predictions.append(near["y"] if label_map is None else label_map[near["donor"]])
    return predictions

def evaluate(train,test,label_map=None):
    predictions = predict_1nn(train,test,label_map)
    truth = [r["y"] if label_map is None else label_map[r["donor"]] for r in test]
    by_donor={}
    for row,p,y in zip(test,predictions,truth):
        by_donor.setdefault(row["donor"], {"pred":[],"truth":y})["pred"].append(p)
    # Predeclared tie rule: class 1 when exactly half the rows predict class 1.
    donor_pairs=[(int(mean(v["pred"]) >= 0.5),v["truth"]) for v in by_donor.values()]
    return {"row_accuracy":mean([p==y for p,y in zip(predictions,truth)]),
            "donor_accuracy":mean([p==y for p,y in donor_pairs]),
            "test_donors":len(by_donor), "test_rows":len(test)}

assert not ({r["donor"] for r in group_train} & {r["donor"] for r in group_test})
within = evaluate(row_train,row_test)
new_donor = evaluate(group_train,group_test)
# Conditional permutation: shuffle labels among TRAIN donors and TEST donors separately,
# preserving each partition's class counts and this predeclared stratified split.
# Every row for one donor retains the same label. This is NOT a cell-wise permutation.
train_ids=sorted({r["donor"] for r in group_train}); test_ids=sorted(holdout)
null=[]
for _ in range(100):
    tr=[labels[i] for i in train_ids]; te=[labels[i] for i in test_ids]
    rng.shuffle(tr); rng.shuffle(te)
    perm=dict(list(zip(train_ids,tr))+list(zip(test_ids,te)))
    null.append(evaluate(group_train,group_test,perm)["donor_accuracy"])
RESULTS = {"data_status": "SYNTHETIC random donor labels and donor signatures", "within_donor_split":within,
           "new_donor_split":new_donor,"conditional_permutation_mean":mean(null),
           "permutation_min_max":[min(null),max(null)],"held_out_donors":sorted(holdout)}
print(json.dumps(RESULTS,indent=2))
optional_plot(["Within-donor", "New-donor", "Permutation mean"],
 [within["donor_accuracy"],new_donor["donor_accuracy"],mean(null)],
 "Donor accuracy (0-1)", "D2 synthetic leakage demonstration")

## Explain the evidence (10 min)
**Q1.** What future population does each split actually evaluate, and where is leakage for the new-donor claim?

**Q2.** Report both metrics, donor counts and the null reference. Why is a four-donor test set too small for a stable clinical conclusion?

**Q3.** Propose a revised external evaluation with preprocessing, model selection and batch separation handled correctly.

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, restart the runtime and run all. Download the `.ipynb` and generated summary JSON. In Colab the JSON is in the Files sidebar. Upload both to the course LMS assignment. Do not email patient data. A completion flag checks presence of responses, not scientific correctness. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":9, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W09_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

## Paper / device-free route
Use supplied sample outputs from the demo notebook solution as labelled simulation results: known-donor 1.00, new-donor value depends on seed. Explain the split diagram and count four independent holdout donors. Do not invent a real diagnostic study.

## Optional extension
Optional: nested cross-validation and calibration; no neural-network training is required.

## Sources
- [S11] scikit-learn: Common pitfalls and recommended practices. https://scikit-learn.org/stable/common_pitfalls.html
- [S13] AnnData documentation: annotated data matrices. https://anndata.readthedocs.io/en/stable/